# 22. Semantic Bridge Pilot v1

이 Notebook은 `docs/plans/SEMANTIC_BRIDGE_PILOT_PLAN.md`의 승인된 조건을 그대로 실행한다.
목적은 간접 향 표현을 제한된 기존 vocabulary에 연결하고, 고정된 11번 Retrieval에서
`Direct-only`와 `Direct + Bridge`를 비교할 수 있는 사람 평가 자료를 만드는 것이다.

실행 상태는 다음 순서를 강제한다.

1. `VALIDATING_PRESELECTED_QUERIES`: 사람이 LLM 실행 전에 고정한 12개 ID를 survey 원문과 대조하고 annotation을 검증한다.
2. `READY_FOR_LLM`: PURE 6/MIXED 6과 direct/bridge 분리가 검증된 뒤 명시적 실행 gate를 기다린다.
3. `WAITING_FOR_HUMAN_EVALUATION`: proposer와 Retrieval 실행 후 blind concept/retrieval 평가표를 사람이 채운다.
4. `COMPLETE`: 사전 고정 지표와 GO/REVISE/STOP 판정을 계산한다.

이 12개는 접근 가치 판단을 위한 목적 표집 Pilot이며 전체 survey 성능을 추정하는 표본이 아니다.
기존 Golden Set, survey 원본, Final Holdout과 원본 perfume data는 읽기만 한다.


In [1]:
import array
import hashlib
import json
import math
import os
import pathlib
import random
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

ROOT = pathlib.Path.cwd()
PLAN_PATH = ROOT / "docs" / "plans" / "SEMANTIC_BRIDGE_PILOT_PLAN.md"
OUTPUT_DIR = ROOT / "analysis_outputs"
EVAL_DIR = ROOT / "evaluation_data" / "semantic_bridge"
OUTPUT_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "plan": PLAN_PATH,
    "survey_candidates": ROOT / "data" / "survey" / "processed" / "survey_nlp_queries_candidates.csv",
    "selected_queries": ROOT / "evaluation_data" / "semantic_bridge" / "22_selected_pilot_queries.csv",
    "selected_queries": ROOT / "evaluation_data" / "semantic_bridge" / "22_selected_pilot_queries.csv",
    "selected_queries": ROOT / "evaluation_data" / "semantic_bridge" / "22_selected_pilot_queries.csv",
    "selected_queries": ROOT / "evaluation_data" / "semantic_bridge" / "22_selected_pilot_queries.csv",
    "accord_dictionary": OUTPUT_DIR / "10_accord_dictionary.csv",
    "note_dictionary": OUTPUT_DIR / "10_note_dictionary.csv",
    "accord_ifra_matching": OUTPUT_DIR / "17_accord_ifra_matching.csv",
    "note_ifra_matching": OUTPUT_DIR / "17_note_ifra_matching.csv",
    "canonical_map": ROOT / "data" / "scent_knowledge" / "fragrantica_note_canonical_map_v1.csv",
    "scent_dictionary": ROOT / "data" / "scent_knowledge" / "scent_term_dictionary_v0.3.csv",
    "scent_evidence": ROOT / "data" / "scent_knowledge" / "scent_term_evidence_v0.3.csv",
    "ifra_definitions": ROOT / "data" / "external" / "ifra" / "processed" / "ifra_primary_descriptor_definitions_2020.csv",
    "perfumes": ROOT / "perfumes.jsonl",
}
missing = [str(path) for path in INPUTS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")

GLOSSARY_PATH = EVAL_DIR / "22_feature_glossary.csv"
HUMAN_EVAL_PATH = EVAL_DIR / "22_pilot_human_evaluation.csv"
PREREG_PATH = OUTPUT_DIR / "22_semantic_bridge_pilot_preregistration.json"
CHECKPOINT_PATH = OUTPUT_DIR / "22_semantic_bridge_pilot_checkpoint.json"
RESULTS_PATH = OUTPUT_DIR / "22_semantic_bridge_pilot_results.json"

PILOT_VERSION = "semantic-bridge-pilot-v1"
BLIND_SALT = "semantic-bridge-pilot-v1-blind|"
TOP_K = 5
MODEL_ID = "gpt-5.4-nano"
ENDPOINT = "https://gms.ssafy.io/gmsapi/api.openai.com/v1/chat/completions"
TEMPERATURE = 0
MAX_RETRIES = 2
REQUEST_TIMEOUT_SECONDS = 90
RUN_LLM_ENABLED = True  # 사전 선정 검증 보고 후 승인되어 고정 12개 실행.


def file_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_hash(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def write_json(path, value):
    path = pathlib.Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    temp_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temp_path.replace(path)


input_hashes = {name: file_sha256(path) for name, path in INPUTS.items()}
print(f"Pilot version: {PILOT_VERSION}")
print(f"Plan SHA256: {input_hashes['plan']}")


Pilot version: semantic-bridge-pilot-v1
Plan SHA256: cb3511d7254afddcf9559daba40bb43d60f974b4692035cd071525fcfd5c1da5


## 1. Allowed vocabulary와 평가자 glossary 동결

- Accord는 10번의 92개를 그대로 사용한다.
- Note는 17번 `STRICT_MATCH` raw Note와 v0.3 `SAME_CONCEPT` term의 합집합을
  21번 canonical map으로 정리한 178개 concept만 허용한다.
- 검색 확장은 canonical map의 `CANONICAL`/`SAME_CONCEPT` raw Note만 사용한다.
- glossary 설명은 기존 IFRA 정의와 검증된 v0.3 evidence만 재사용한다.
  정의가 없는 feature에는 새 설명을 만들지 않는다.


In [2]:
accord_dictionary_df = pd.read_csv(INPUTS["accord_dictionary"])
note_dictionary_df = pd.read_csv(INPUTS["note_dictionary"])
accord_ifra_df = pd.read_csv(INPUTS["accord_ifra_matching"])
note_ifra_df = pd.read_csv(INPUTS["note_ifra_matching"])
canonical_map_df = pd.read_csv(INPUTS["canonical_map"])
scent_dictionary_df = pd.read_csv(INPUTS["scent_dictionary"])
scent_evidence_df = pd.read_csv(INPUTS["scent_evidence"])
ifra_definitions_df = pd.read_csv(INPUTS["ifra_definitions"])

allowed_accords = sorted(accord_dictionary_df["accord"].dropna().astype(str).unique())

strict_raw_notes = set(
    note_ifra_df.loc[
        note_ifra_df["final_auto_status"].eq("STRICT_MATCH"), "note"
    ].dropna().astype(str)
)
strict_raw_notes.update(
    scent_dictionary_df.loc[
        scent_dictionary_df["relation_type"].eq("SAME_CONCEPT"), "term"
    ].dropna().astype(str)
)
allowed_canonical_notes = sorted(
    canonical_map_df.loc[
        canonical_map_df["raw_note"].isin(strict_raw_notes), "canonical_note"
    ].dropna().astype(str).unique()
)

safe_map_df = canonical_map_df[
    canonical_map_df["relation"].isin(["CANONICAL", "SAME_CONCEPT"])
].copy()
safe_note_expansions = {
    canonical: sorted(group["raw_note"].dropna().astype(str).unique())
    for canonical, group in safe_map_df.groupby("canonical_note", sort=False)
    if canonical in set(allowed_canonical_notes)
}

assert len(allowed_accords) == 92, len(allowed_accords)
assert len(strict_raw_notes) == 185, len(strict_raw_notes)
assert len(allowed_canonical_notes) == 178, len(allowed_canonical_notes)
assert set(safe_map_df["relation"]).issubset({"CANONICAL", "SAME_CONCEPT"})

allowed_vocabulary = [
    {"target_type": "ACCORD", "target_feature": name}
    for name in allowed_accords
] + [
    {"target_type": "CANONICAL_NOTE", "target_feature": name}
    for name in allowed_canonical_notes
]
allowed_vocabulary_hash = json_hash(allowed_vocabulary)

definition_by_descriptor = {
    str(row.descriptor).casefold(): str(row.definition)
    for row in ifra_definitions_df.itertuples(index=False)
    if pd.notna(row.descriptor) and pd.notna(row.definition)
}

accord_match_by_name = accord_ifra_df.set_index("accord", drop=False).to_dict("index")
evidence_same = scent_evidence_df[
    scent_evidence_df["supports_relation"].eq("SAME_CONCEPT")
].copy()


def first_verified_note_description(canonical_note, aliases):
    matched_terms = note_ifra_df.loc[
        note_ifra_df["note"].isin(aliases)
        & note_ifra_df["final_auto_status"].eq("STRICT_MATCH"),
        "strict_matched_term",
    ].dropna().astype(str)
    for term in matched_terms:
        definition = definition_by_descriptor.get(term.casefold())
        if definition:
            return definition, "IFRA primary descriptor definition"
    evidence = evidence_same[
        evidence_same["term"].isin(aliases)
        | evidence_same["candidate_term"].eq(canonical_note)
    ]["evidence_summary"].dropna().astype(str)
    if not evidence.empty:
        return evidence.iloc[0], "scent_term_evidence_v0.3 SAME_CONCEPT"
    return "", ""


glossary_rows = []
for accord in allowed_accords:
    match = accord_match_by_name.get(accord, {})
    matched = str(match.get("matched_ifra_term") or "")
    definition = definition_by_descriptor.get(matched.casefold(), "") if matched else ""
    glossary_rows.append({
        "feature_type": "ACCORD",
        "feature_name": accord,
        "verified_definition": definition,
        "same_concept_aliases_json": "[]",
        "definition_source": "IFRA primary descriptor definition" if definition else "",
    })
for canonical_note in allowed_canonical_notes:
    aliases = safe_note_expansions.get(canonical_note, [])
    definition, source = first_verified_note_description(canonical_note, aliases)
    glossary_rows.append({
        "feature_type": "CANONICAL_NOTE",
        "feature_name": canonical_note,
        "verified_definition": definition,
        "same_concept_aliases_json": json.dumps(aliases, ensure_ascii=False),
        "definition_source": source,
    })

expected_glossary_df = pd.DataFrame(glossary_rows).sort_values(
    ["feature_type", "feature_name"], kind="stable"
).reset_index(drop=True)
if GLOSSARY_PATH.exists():
    existing_glossary_df = pd.read_csv(GLOSSARY_PATH, keep_default_na=False)
    pd.testing.assert_frame_equal(existing_glossary_df, expected_glossary_df)
else:
    expected_glossary_df.to_csv(GLOSSARY_PATH, index=False, encoding="utf-8-sig")
glossary_hash = file_sha256(GLOSSARY_PATH)
glossary_df = pd.read_csv(GLOSSARY_PATH, keep_default_na=False)

print(f"Allowed vocabulary: Accord {len(allowed_accords)} + Canonical Note {len(allowed_canonical_notes)}")
print(f"Allowed vocabulary SHA256: {allowed_vocabulary_hash}")
print(f"Glossary: {GLOSSARY_PATH} ({len(glossary_df)} rows, SHA256={glossary_hash})")


Allowed vocabulary: Accord 92 + Canonical Note 178
Allowed vocabulary SHA256: a7b2c1babddc07d8ab1c0222704602dd281d87c0e3c5d353ef0e2b49fb56e7d8
Glossary: C:\Users\SSAFY\Desktop\EDA\evaluation_data\semantic_bridge\22_feature_glossary.csv (270 rows, SHA256=7b6e353ccccf5f60979f0f5c9dbbadca9e440cd0b0ce5f0c7e2ea3437aee0bab)


In [3]:
SYSTEM_PROMPT = f'''You are the sole candidate proposer for Semantic Bridge Pilot v1.
Read only the Korean bridge_phrase supplied by the user. Select zero to three scent targets
from the exact closed vocabulary below. Do not recommend perfumes. Do not modify direct,
context, or avoid conditions. Do not invent targets or silently correct names. If the closed
vocabulary does not support a defensible mapping, return ABSTAIN. Use REVIEW only when the
phrase appears scent-related but is too ambiguous to apply automatically.

Return exactly one JSON object with these keys and no prose:
{{"bridge_status":"MAPPED|ABSTAIN|REVIEW","targets":[{{"target_type":"ACCORD|CANONICAL_NOTE","target_feature":"exact allowed name"}}],"abstain_reason":"short text or empty string"}}

Rules:
- MAPPED requires 1-3 unique exact targets.
- ABSTAIN and REVIEW require an empty targets list.
- Never output confidence, weights, rationale, aliases, perfume names, or extra keys.

ALLOWED ACCORD ({len(allowed_accords)}):
{json.dumps(allowed_accords, ensure_ascii=False)}

ALLOWED CANONICAL_NOTE ({len(allowed_canonical_notes)}):
{json.dumps(allowed_canonical_notes, ensure_ascii=False)}'''
USER_PROMPT_TEMPLATE = 'bridge_phrase: {bridge_phrase}'
prompt_hash = json_hash({
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt_template": USER_PROMPT_TEMPLATE,
})

preregistration_base = {
    "pilot_version": PILOT_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_plan": str(PLAN_PATH.relative_to(ROOT)),
    "source_plan_sha256": input_hashes["plan"],
    "protected_inputs_sha256": input_hashes,
    "selection": {
        "method": "MANUAL_PRESELECTED",
        "source_path": str(INPUTS["selected_queries"].relative_to(ROOT)),
        "source_sha256": input_hashes["selected_queries"],
        "population_reference": str(INPUTS["survey_candidates"].relative_to(ROOT)),
        "expected_query_count": 12,
        "expected_group_counts": {"PURE": 6, "MIXED": 6},
        "replacement_rule": "No automatic or discretionary replacement; report and stop on invalid query",
        "generalization_limit": "Purposeful feasibility pilot; do not generalize to all 155 survey queries",
        "selected_queries": [],
    },
    "allowed_vocabulary": {
        "accord_count": len(allowed_accords),
        "canonical_note_count": len(allowed_canonical_notes),
        "strict_raw_note_count": len(strict_raw_notes),
        "sha256": allowed_vocabulary_hash,
        "note_equivalence_relations": ["CANONICAL", "SAME_CONCEPT"],
        "excluded_relations": ["FAMILY", "RELATED", "REVIEW", "NOT_SAME"],
        "items": allowed_vocabulary,
    },
    "feature_glossary": {
        "path": str(GLOSSARY_PATH.relative_to(ROOT)),
        "rows": len(glossary_df),
        "sha256": glossary_hash,
        "generated_without_llm": True,
    },
    "llm": {
        "model": MODEL_ID,
        "endpoint": ENDPOINT,
        "temperature": TEMPERATURE,
        "max_retries_after_initial": MAX_RETRIES,
        "timeout_seconds": REQUEST_TIMEOUT_SECONDS,
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt_template": USER_PROMPT_TEMPLATE,
        "prompt_sha256": prompt_hash,
        "one_response_per_query": True,
    },
    "validation": {
        "statuses": ["MAPPED", "ABSTAIN", "REVIEW"],
        "target_count": "1-3 only for MAPPED; 0 for ABSTAIN/REVIEW",
        "exact_vocabulary_match": True,
        "duplicates_allowed": False,
        "avoid_conflict_action": "REVIEW and do not retrieve Bridge targets",
        "automatic_spelling_correction": False,
    },
    "retrieval": {
        "baseline": "Direct Accord/Canonical Note/Season/Daypart only",
        "treatment": "Baseline plus validated Bridge Accord/Canonical Note",
        "score": "11_rule_based_query_retrieval_baseline category mean times evidence coverage",
        "tie_break": ["final_score desc", "evidence_coverage desc", "base_match_score desc", "perfume_id asc"],
        "top_k": TOP_K,
        "canonical_note_match": "OR across CANONICAL/SAME_CONCEPT raw aliases",
    },
    "human_evaluation": {
        "independent_reviewers": 2,
        "blind_salt": BLIND_SALT,
        "concept_scale": {"2": "STRONG", "1": "PLAUSIBLE", "0": "WRONG/HARMFUL"},
        "retrieval_scale": {"2": "HIGHLY_RELEVANT", "1": "PARTIALLY_RELEVANT", "0": "NOT_RELEVANT"},
        "abstain_causes": ["APPROPRIATE_ABSTENTION", "TARGET_SPACE_LIMITATION", "MAPPING_FAILURE"],
        "agreement": "exact agreement and quadratic weighted Cohen's kappa; constant identical ratings count as kappa 1.0",
        "kappa_gate": 0.40,
        "pooled_ndcg_scope": "Baseline/Treatment TOP 5 union only; not global corpus judgment",
    },
    "decision_thresholds": {
        "go": {
            "safe_resolution_min": 9,
            "query_count": 12,
            "graded_concept_precision_at_3_min": 0.67,
            "wrong_harmful_rate_max": 0.20,
            "mixed_wins_min": 4,
            "mixed_mean_delta_pooled_ndcg_positive": True,
            "pure_strong_hit_queries_min": 4,
        },
        "stop": {
            "safe_resolution_below": 6,
            "graded_concept_precision_at_3_below": 0.50,
            "wrong_harmful_rate_above": 0.30,
            "operational_no_retrieval_value": "mixed mean delta <= 0 and pure strong-hit count = 0",
        },
    },
    "protections": {
        "final_holdout_used": False,
        "golden_set_modified": False,
        "survey_source_modified": False,
        "new_external_data_added": False,
    },
    "state": "VALIDATING_PRESELECTED_QUERIES",
}


def immutable_prereg_view(value):
    return {key: value[key] for key in [
        "pilot_version", "source_plan", "source_plan_sha256",
        "protected_inputs_sha256", "allowed_vocabulary", "feature_glossary",
        "llm", "validation", "retrieval", "human_evaluation",
        "decision_thresholds", "protections",
    ]}


if PREREG_PATH.exists():
    preregistration = json.loads(PREREG_PATH.read_text(encoding="utf-8"))
    if immutable_prereg_view(preregistration) != immutable_prereg_view(preregistration_base):
        raise RuntimeError("기존 preregistration의 고정 조건이 현재 코드/입력과 다릅니다. 덮어쓰지 않습니다.")
else:
    preregistration = preregistration_base
    write_json(PREREG_PATH, preregistration)

print(f"Prompt SHA256: {prompt_hash}")
print(f"Preregistration: {PREREG_PATH}")


Prompt SHA256: 1dd2c0309d290ecc03a7eff656874e867e456e7393bbb4ec175a81fcdcf9c19e
Preregistration: C:\Users\SSAFY\Desktop\EDA\analysis_outputs\22_semantic_bridge_pilot_preregistration.json


## 2. 사람이 사전 고정한 12개 Query 검증

`evaluation_data/semantic_bridge/22_selected_pilot_queries.csv`의 12개 ID를 이번 Pilot 입력으로 사용한다.
Query 원문은 selected CSV에 복사하지 않고 `survey_nlp_queries_candidates.csv`에서 `query_id`로 가져온다.

실행 전에 다음을 강제한다.

- unique ID 12개와 `PURE` 6 / `MIXED` 6
- 모든 ID의 survey 원본 존재
- PURE에는 positive direct Accord/Canonical Note가 없고 indirect scent phrase가 있음
- MIXED에는 allowed vocabulary의 positive direct Accord/Canonical Note가 최소 하나 있고 indirect scent phrase가 있음
- direct/context/avoid 근거 span과 `bridge_phrase`가 실제 원문의 exact span
- 부적격 Query가 있으면 자동 또는 임의 교체 없이 중단

이 검증은 선정된 사례의 구조 적합성만 확인한다. 155개 전체 screening이나 hash selection을 수행하지 않으며,
이 12개의 결과를 전체 survey 성능으로 일반화하지 않는다.


In [4]:
survey_df = pd.read_csv(INPUTS["survey_candidates"], keep_default_na=False)
selected_input_df = pd.read_csv(INPUTS["selected_queries"], keep_default_na=False)

expected_columns = ["query_id", "group", "selection_reason"]
if selected_input_df.columns.tolist() != expected_columns:
    raise RuntimeError(f"Selected CSV schema는 {expected_columns}여야 합니다: {selected_input_df.columns.tolist()}")
if len(survey_df) != 155 or survey_df["query_id"].duplicated().any():
    raise RuntimeError("Survey 모집단은 unique query_id 155개여야 합니다.")
if len(selected_input_df) != 12 or selected_input_df["query_id"].duplicated().any():
    raise RuntimeError("Selected CSV는 unique query_id 12개여야 합니다.")
if selected_input_df["group"].value_counts().to_dict() != {"PURE": 6, "MIXED": 6}:
    raise RuntimeError(f"Selected group은 PURE 6/MIXED 6이어야 합니다: {selected_input_df['group'].value_counts().to_dict()}")
if (selected_input_df["selection_reason"].astype(str).str.strip() == "").any():
    raise RuntimeError("모든 selected Query에 selection_reason이 필요합니다.")

missing_query_ids = sorted(set(selected_input_df["query_id"]) - set(survey_df["query_id"]))
if missing_query_ids:
    raise RuntimeError(f"Survey candidate에 없는 selected query_id가 있습니다: {missing_query_ids}")

selected_source_df = selected_input_df.merge(
    survey_df[["query_id", "query_text", "source", "source_row"]],
    on="query_id", how="left", validate="one_to_one",
)
print(f"Selected input: {INPUTS['selected_queries']} ({len(selected_source_df)} rows)")
print(f"Survey ID match: {selected_source_df['query_text'].notna().sum()}/12")
print(f"Group counts: {selected_source_df['group'].value_counts().to_dict()}")


Selected input: C:\Users\SSAFY\Desktop\EDA\evaluation_data\semantic_bridge\22_selected_pilot_queries.csv (12 rows)
Survey ID match: 12/12
Group counts: {'PURE': 6, 'MIXED': 6}


In [5]:
EXPRESSION_TYPES = {
    "NATURAL_ENVIRONMENT",
    "CLEAN_FABRIC_ROUTINE",
    "TEMPERATURE_TEXTURE_ATMOSPHERE",
}
SEASONS = {"winter", "spring", "summer", "autumn"}
DAYPARTS = {"day", "night"}

def clean(value):
    return "" if value is None else str(value).strip()

# LLM output을 보기 전에 고정한 direct/context/avoid와 exact-span bridge annotation.
# direct_evidence_spans/context_evidence_spans/avoid_evidence_spans는 내용 검증용이며 Retrieval score에는 직접 쓰지 않는다.
PRESELECTED_ANNOTATIONS = {
    "SQ0061": {
        "primary_expression_type": "NATURAL_ENVIRONMENT", "secondary_expression_types": [],
        "direct_accords": [], "direct_canonical_notes": [], "seasons": [], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": [], "context_evidence_spans": [], "avoid_evidence_spans": [],
        "bridge_phrase": "비 온 뒤의 숲의 냄새", "bridge_evaluation_text": "비 온 뒤의 숲의 냄새",
        "excluded_other_requirements": "", "bridge_applicability": "YES",
    },
    "SQ0032": {
        "primary_expression_type": "NATURAL_ENVIRONMENT", "secondary_expression_types": [],
        "direct_accords": [], "direct_canonical_notes": [], "seasons": ["summer"], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": [], "context_evidence_spans": ["여름"], "avoid_evidence_spans": [],
        "bridge_phrase": "숲 속에 온 듯한 향", "bridge_evaluation_text": "여름에 어울리는 숲 속에 온 듯한 향",
        "excluded_other_requirements": "", "bridge_applicability": "YES",
    },
    "SQ0136": {
        "primary_expression_type": "CLEAN_FABRIC_ROUTINE", "secondary_expression_types": [],
        "direct_accords": [], "direct_canonical_notes": [], "seasons": [], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": [], "context_evidence_spans": [], "avoid_evidence_spans": [],
        "bridge_phrase": "빨래향 같이 자연스러운 향", "bridge_evaluation_text": "빨래향 같이 자연스러운 향",
        "excluded_other_requirements": "향이 너무 강하지 않음; 강한 향으로 인한 두통", "bridge_applicability": "YES",
    },
    "SQ0105": {
        "primary_expression_type": "TEMPERATURE_TEXTURE_ATMOSPHERE", "secondary_expression_types": [],
        "direct_accords": [], "direct_canonical_notes": [], "seasons": [], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": [], "context_evidence_spans": [], "avoid_evidence_spans": [],
        "bridge_phrase": "따뜻한 햇살을 받으며 침대 위에 누워있는 듯한 느낌의 향수",
        "bridge_evaluation_text": "따뜻한 햇살을 받으며 침대 위에 누워있는 듯한 느낌의 향수",
        "excluded_other_requirements": "주말 오후 사용 상황", "bridge_applicability": "UNCERTAIN",
    },
    "SQ0012": {
        "primary_expression_type": "CLEAN_FABRIC_ROUTINE", "secondary_expression_types": ["TEMPERATURE_TEXTURE_ATMOSPHERE"],
        "direct_accords": [], "direct_canonical_notes": [], "seasons": [], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": [], "context_evidence_spans": [], "avoid_evidence_spans": [],
        "bridge_phrase": "샤워하고 나온듯한 따뜻하고 포근한 느낌의 향기",
        "bridge_evaluation_text": "샤워하고 나온듯한 따뜻하고 포근한 느낌의 향기",
        "excluded_other_requirements": "", "bridge_applicability": "YES",
    },
    "SQ0058": {
        "primary_expression_type": "NATURAL_ENVIRONMENT", "secondary_expression_types": ["TEMPERATURE_TEXTURE_ATMOSPHERE"],
        "direct_accords": [], "direct_canonical_notes": [], "seasons": ["summer"], "dayparts": [],
        "avoid_accords": ["woody", "musky"], "avoid_canonical_notes": [],
        "direct_evidence_spans": [], "context_evidence_spans": ["여름"], "avoid_evidence_spans": ["우드", "머스크"],
        "bridge_phrase": "바다가 생각나는 시원한 향", "bridge_evaluation_text": "여름 바다가 생각나는 시원한 향",
        "excluded_other_requirements": "가격 10만원 미만; 지속력; 성별; 너무 진한 향 회피", "bridge_applicability": "YES",
    },
    "SQ0002": {
        "primary_expression_type": "TEMPERATURE_TEXTURE_ATMOSPHERE", "secondary_expression_types": [],
        "direct_accords": ["citrus"], "direct_canonical_notes": [], "seasons": [], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": ["시트러스"], "context_evidence_spans": [], "avoid_evidence_spans": [],
        "bridge_phrase": "시원하고 깔끔한 향", "bridge_evaluation_text": "시트러스 향처럼 시원하고 깔끔한 향",
        "excluded_other_requirements": "", "bridge_applicability": "UNCERTAIN",
    },
    "SQ0047": {
        "primary_expression_type": "TEMPERATURE_TEXTURE_ATMOSPHERE", "secondary_expression_types": [],
        "direct_accords": ["soapy", "musky", "woody"], "direct_canonical_notes": [], "seasons": ["summer"], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": ["비누향", "머스크향", "우드향"], "context_evidence_spans": ["여름"], "avoid_evidence_spans": [],
        "bridge_phrase": "가벼우면서 시원한 느낌",
        "bridge_evaluation_text": "여름에 쓰기 좋은 가벼우면서 시원한 느낌의 비누향, 머스크향, 약간의 우드향",
        "excluded_other_requirements": "", "bridge_applicability": "YES",
    },
    "SQ0051": {
        "primary_expression_type": "CLEAN_FABRIC_ROUTINE", "secondary_expression_types": ["TEMPERATURE_TEXTURE_ATMOSPHERE"],
        "direct_accords": [], "direct_canonical_notes": ["Peach"], "seasons": [], "dayparts": [],
        "avoid_accords": ["soapy"], "avoid_canonical_notes": [],
        "direct_evidence_spans": ["복숭아 향"], "context_evidence_spans": [], "avoid_evidence_spans": ["비누향이 아닌"],
        "bridge_phrase": "톡 쏘는 듯한 청량함과 방금 막 세탁된 코튼 향처럼 깨끗한 공기향",
        "bridge_evaluation_text": "톡 쏘는 듯한 청량함과 방금 막 세탁된 코튼 향처럼 깨끗한 공기향, 복숭아 향",
        "excluded_other_requirements": "향 강도; 니치 향수 특유의 깊고 크리미한 잔향; 3개 추천과 설명 요청",
        "bridge_applicability": "YES",
    },
    "SQ0071": {
        "primary_expression_type": "TEMPERATURE_TEXTURE_ATMOSPHERE", "secondary_expression_types": ["NATURAL_ENVIRONMENT"],
        "direct_accords": ["musky", "woody"], "direct_canonical_notes": [], "seasons": ["winter"], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": ["머스크 계열", "우디한"], "context_evidence_spans": ["겨울"], "avoid_evidence_spans": [],
        "bridge_phrase": "겨울 아침에 딱~ 일어나서 창문을 열었는데 느껴지는 차갑고 상쾌한 향",
        "bridge_evaluation_text": "겨울 아침의 차갑고 상쾌한 향, 머스크 계열, 우디한 향",
        "excluded_other_requirements": "탑/미들/베이스 시간대별 전개와 무게감", "bridge_applicability": "YES",
    },
    "SQ0073": {
        "primary_expression_type": "NATURAL_ENVIRONMENT", "secondary_expression_types": ["TEMPERATURE_TEXTURE_ATMOSPHERE"],
        "direct_accords": ["woody", "floral"], "direct_canonical_notes": [], "seasons": [], "dayparts": [],
        "avoid_accords": ["fresh spicy", "warm spicy"], "avoid_canonical_notes": [],
        "direct_evidence_spans": ["우디", "플로럴"], "context_evidence_spans": [], "avoid_evidence_spans": ["스파이시한 계열"],
        "bridge_phrase": "깨끗하고 시원한 향을 좋아하고, 편백나무, 산내음처럼 자연스러운 향",
        "bridge_evaluation_text": "깨끗하고 시원한 향, 편백나무와 산내음처럼 자연스러운 향, 우디와 플로럴 계열",
        "excluded_other_requirements": "", "bridge_applicability": "YES",
    },
    "SQ0132": {
        "primary_expression_type": "TEMPERATURE_TEXTURE_ATMOSPHERE", "secondary_expression_types": [],
        "direct_accords": ["musky"], "direct_canonical_notes": [], "seasons": [], "dayparts": [],
        "avoid_accords": [], "avoid_canonical_notes": [],
        "direct_evidence_spans": ["머스크 계열"], "context_evidence_spans": [], "avoid_evidence_spans": [],
        "bridge_phrase": "포근한 느낌", "bridge_evaluation_text": "머스크 계열 향과 포근한 느낌",
        "excluded_other_requirements": "가벼운 무게감; 오래 남는 잔향; 상큼한 향도 허용", "bridge_applicability": "UNCERTAIN",
    },
}

validation_errors = []
if set(PRESELECTED_ANNOTATIONS) != set(selected_source_df["query_id"]):
    validation_errors.append("Selected ID와 annotation ID가 일치하지 않습니다.")

selected_records = []
for row in selected_source_df.to_dict("records"):
    annotation = PRESELECTED_ANNOTATIONS.get(row["query_id"])
    if annotation is None:
        continue
    query_text = row["query_text"]
    direct_count = len(annotation["direct_accords"]) + len(annotation["direct_canonical_notes"])
    if row["group"] == "PURE" and direct_count != 0:
        validation_errors.append(f"{row['query_id']}: PURE에 positive direct scent feature가 있습니다.")
    if row["group"] == "MIXED" and direct_count == 0:
        validation_errors.append(f"{row['query_id']}: MIXED에 positive direct scent feature가 없습니다.")
    if annotation["bridge_phrase"] not in query_text:
        validation_errors.append(f"{row['query_id']}: bridge_phrase가 원문의 exact span이 아닙니다.")
    for span_type in ["direct_evidence_spans", "context_evidence_spans", "avoid_evidence_spans"]:
        for span in annotation[span_type]:
            if span not in query_text:
                validation_errors.append(f"{row['query_id']}: {span_type}의 {span!r}이 원문에 없습니다.")
    if not set(annotation["direct_accords"] + annotation["avoid_accords"]).issubset(set(allowed_accords)):
        validation_errors.append(f"{row['query_id']}: Accord가 allowed vocabulary 밖입니다.")
    if not set(annotation["direct_canonical_notes"] + annotation["avoid_canonical_notes"]).issubset(set(allowed_canonical_notes)):
        validation_errors.append(f"{row['query_id']}: Canonical Note가 allowed vocabulary 밖입니다.")
    if not set(annotation["seasons"]).issubset(SEASONS) or not set(annotation["dayparts"]).issubset(DAYPARTS):
        validation_errors.append(f"{row['query_id']}: context가 허용값 밖입니다.")
    if annotation["primary_expression_type"] not in EXPRESSION_TYPES:
        validation_errors.append(f"{row['query_id']}: primary expression type 오류")
    if not set(annotation["secondary_expression_types"]).issubset(EXPRESSION_TYPES):
        validation_errors.append(f"{row['query_id']}: secondary expression type 오류")
    if annotation["bridge_applicability"] not in {"YES", "UNCERTAIN", "NO"}:
        validation_errors.append(f"{row['query_id']}: bridge_applicability 오류")

    selected_records.append({
        "query_id": row["query_id"], "query_text": query_text,
        "source": row["source"], "source_row": row["source_row"],
        "resolution_type": row["group"], "selection_reason": row["selection_reason"],
        **annotation,
    })

selection_validated = len(validation_errors) == 0 and len(selected_records) == 12
if not selection_validated:
    display(pd.DataFrame({"validation_error": validation_errors}))
    raise RuntimeError("사전 선정 Query 검증에 실패했습니다. Query를 교체하지 않고 중단합니다.")

selected_queries_df = pd.DataFrame(selected_records)
print("Preselected Query validation: PASS")
display(selected_queries_df[[
    "query_id", "resolution_type", "selection_reason", "direct_accords",
    "direct_canonical_notes", "seasons", "dayparts", "avoid_accords",
    "bridge_phrase", "bridge_applicability",
]])


Preselected Query validation: PASS


,query_id,resolution_type,selection_reason,direct_accords,direct_canonical_notes,seasons,dayparts,avoid_accords,bridge_phrase,bridge_applicability
0,SQ0061,PURE,natural_environment,[],[],[],[],[],비 온 뒤의 숲의 냄새,YES
1,SQ0032,PURE,natural_environment,[],[],[summer],[],[],숲 속에 온 듯한 향,YES
2,SQ0136,PURE,clean_fabric_routine,[],[],[],[],[],빨래향 같이 자연스러운 향,YES
3,SQ0105,PURE,temperature_texture_atmosphere,[],[],[],[],[],따뜻한 햇살을 받으며 침대 위에 누워있는 듯한 느낌의 향수,UNCERTAIN
4,SQ0012,PURE,clean_fabric_routine,[],[],[],[],[],샤워하고 나온듯한 따뜻하고 포근한 느낌의 향기,YES
5,SQ0058,PURE,natural_environment,[],[],[summer],[],"[woody, musky]",바다가 생각나는 시원한 향,YES
6,SQ0002,MIXED,mixed_indirect_and_direct,[citrus],[],[],[],[],시원하고 깔끔한 향,UNCERTAIN
7,SQ0047,MIXED,mixed_indirect_and_direct,"[soapy, musky, woody]",[],[summer],[],[],가벼우면서 시원한 느낌,YES
8,SQ0051,MIXED,clean_fabric_with_direct_feature,[],[Peach],[],[],[soapy],톡 쏘는 듯한 청량함과 방금 막 세탁된 코튼 향처럼 깨끗한 공기향,YES
9,SQ0071,MIXED,temperature_atmosphere_with_direct_feature,"[musky, woody]",[],[winter],[],[],겨울 아침에 딱~ 일어나서 창문을 열었는데 느껴지는 차갑고 상쾌한 향,YES


In [6]:
selected_records = selected_queries_df.replace({np.nan: None}).to_dict("records")
selection_annotation_hash = json_hash(selected_records)
frozen_selection = preregistration["selection"].get("selected_queries", [])
if frozen_selection:
    if preregistration["selection"].get("selection_annotation_sha256") != selection_annotation_hash:
        raise RuntimeError("이미 동결된 12개/annotation과 현재 selected CSV 또는 원문이 다릅니다.")
else:
    preregistration["selection"].update({
        "validated_before_llm": True,
        "validated_at_utc": datetime.now(timezone.utc).isoformat(),
        "survey_source_sha256": input_hashes["survey_candidates"],
        "selected_queries": selected_records,
        "selected_query_ids": selected_queries_df["query_id"].tolist(),
        "actual_group_counts": selected_queries_df["resolution_type"].value_counts().to_dict(),
        "selection_annotation_sha256": selection_annotation_hash,
        "invalid_query_replacements": [],
    })
preregistration["state"] = "READY_FOR_LLM"
preregistration["execution_gate"] = {
    "run_llm_enabled": RUN_LLM_ENABLED,
    "reason": "Report 12-query validation before the first proposer call",
}
write_json(PREREG_PATH, preregistration)
print(f"12개 Query와 annotation을 LLM 실행 전에 동결했습니다. SHA256={selection_annotation_hash}")
print("LLM execution gate: CLOSED" if not RUN_LLM_ENABLED else "LLM execution gate: OPEN")


12개 Query와 annotation을 LLM 실행 전에 동결했습니다. SHA256=6ed45347220cf22c173970e97c6a47d890da402d53cf71255821eafd0d176353
LLM execution gate: OPEN


## 3. Constrained proposer와 exact validation

proposer는 `bridge_phrase`만 읽고 allowed vocabulary에서 최대 3개를 제안한다.
응답은 한 번 checkpoint한 뒤 낮은 평가를 이유로 재호출하지 않는다. API/schema 실패만
사전 고정 횟수 안에서 재시도한다. Exact schema/vocabulary/중복/avoid 검사를 통과하지 못하면
자동 교정하지 않고 `ABSTAIN` 또는 `REVIEW`로 남긴다.


In [7]:
ALLOWED_BY_TYPE = {
    "ACCORD": set(allowed_accords),
    "CANONICAL_NOTE": set(allowed_canonical_notes),
}


def extract_json_object(text):
    text = clean(text)
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return json.loads(text)


def validate_proposer_content(content, selected_row):
    try:
        payload = extract_json_object(content)
        if not isinstance(payload, dict):
            raise ValueError("top-level JSON object가 아닙니다.")
        if set(payload) != {"bridge_status", "targets", "abstain_reason"}:
            raise ValueError("schema key가 exact match가 아닙니다.")
        status = payload["bridge_status"]
        targets = payload["targets"]
        reason = payload["abstain_reason"]
        if status not in {"MAPPED", "ABSTAIN", "REVIEW"}:
            raise ValueError("bridge_status가 허용값이 아닙니다.")
        if not isinstance(targets, list) or not isinstance(reason, str):
            raise ValueError("targets/abstain_reason type 오류")
        if status == "MAPPED" and not 1 <= len(targets) <= 3:
            raise ValueError("MAPPED target 수가 1-3이 아닙니다.")
        if status in {"ABSTAIN", "REVIEW"} and targets:
            raise ValueError("ABSTAIN/REVIEW targets가 비어 있지 않습니다.")
        normalized = []
        for target in targets:
            if not isinstance(target, dict) or set(target) != {"target_type", "target_feature"}:
                raise ValueError("target schema가 exact match가 아닙니다.")
            target_type = target["target_type"]
            target_feature = target["target_feature"]
            if target_type not in ALLOWED_BY_TYPE:
                raise ValueError(f"target_type 오류: {target_type!r}")
            if target_feature not in ALLOWED_BY_TYPE[target_type]:
                raise ValueError(f"vocabulary exact-match 실패: {target_type}/{target_feature}")
            normalized.append({"target_type": target_type, "target_feature": target_feature})
        identities = [(item["target_type"], item["target_feature"]) for item in normalized]
        if len(identities) != len(set(identities)):
            raise ValueError("duplicate target")

        avoid = {
            *(('ACCORD', item) for item in selected_row["avoid_accords"]),
            *(('CANONICAL_NOTE', item) for item in selected_row["avoid_canonical_notes"]),
        }
        conflicts = sorted(set(identities) & avoid)
        if conflicts:
            return {
                "completed": True,
                "raw_bridge_status": status,
                "validated_bridge_status": "REVIEW",
                "validated_targets": [],
                "validation_error": "EXPLICIT_AVOID_CONFLICT",
                "avoid_conflicts": conflicts,
            }
        return {
            "completed": True,
            "raw_bridge_status": status,
            "validated_bridge_status": status,
            "validated_targets": normalized if status == "MAPPED" else [],
            "validation_error": "",
            "avoid_conflicts": [],
        }
    except Exception as error:
        return {
            "completed": True,
            "raw_bridge_status": "INVALID_OUTPUT",
            "validated_bridge_status": "ABSTAIN",
            "validated_targets": [],
            "validation_error": str(error),
            "avoid_conflicts": [],
        }


# 기계적 validation 자체는 외부 호출 없이 검증한다.
_test_row = {"avoid_accords": [], "avoid_canonical_notes": []}
_test_valid = json.dumps({
    "bridge_status": "MAPPED",
    "targets": [{"target_type": "ACCORD", "target_feature": allowed_accords[0]}],
    "abstain_reason": "",
}, ensure_ascii=False)
assert validate_proposer_content(_test_valid, _test_row)["validated_bridge_status"] == "MAPPED"
_test_invalid = json.dumps({
    "bridge_status": "MAPPED",
    "targets": [{"target_type": "ACCORD", "target_feature": "__NOT_ALLOWED__"}],
    "abstain_reason": "",
}, ensure_ascii=False)
assert validate_proposer_content(_test_invalid, _test_row)["validated_bridge_status"] == "ABSTAIN"
print("Proposer output validation self-test: PASS")


Proposer output validation self-test: PASS


In [8]:
def initial_checkpoint():
    return {
        "pilot_version": PILOT_VERSION,
        "selection_annotation_sha256": preregistration["selection"].get("selection_annotation_sha256"),
        "allowed_vocabulary_sha256": allowed_vocabulary_hash,
        "prompt_sha256": prompt_hash,
        "model": MODEL_ID,
        "llm_results": [],
        "retrieval_key": {},
    }


def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return initial_checkpoint()
    checkpoint = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    expected = initial_checkpoint()
    for key in [
        "pilot_version", "selection_annotation_sha256",
        "allowed_vocabulary_sha256", "prompt_sha256", "model",
    ]:
        if checkpoint.get(key) != expected.get(key):
            raise RuntimeError(f"Checkpoint {key}가 preregistration과 다릅니다.")
    return checkpoint


def call_proposer(bridge_phrase, api_key):
    import requests

    response = requests.post(
        ENDPOINT,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={
            "model": MODEL_ID,
            "temperature": TEMPERATURE,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT_TEMPLATE.format(bridge_phrase=bridge_phrase)},
            ],
        },
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    payload = response.json()
    content = payload["choices"][0]["message"]["content"]
    return payload, content


llm_ready = False
checkpoint = None
if selection_validated and RUN_LLM_ENABLED:
    try:
        from dotenv import load_dotenv
        load_dotenv(ROOT / ".env")
    except ImportError:
        pass
    api_key = os.getenv("GMS_KEY", "")
    if not api_key:
        print("WAITING_FOR_API_KEY: GMS_KEY가 없어 외부 호출을 하지 않았습니다.")
    else:
        checkpoint = load_checkpoint()
        completed_by_id = {
            item["query_id"]: item
            for item in checkpoint["llm_results"]
            if item.get("completed")
        }
        selected_by_id = {row["query_id"]: row for row in preregistration["selection"]["selected_queries"]}
        for query_id, selected_row in selected_by_id.items():
            if query_id in completed_by_id:
                continue
            attempts = []
            final_record = None
            for attempt_index in range(MAX_RETRIES + 1):
                try:
                    raw_payload, content = call_proposer(selected_row["bridge_phrase"], api_key)
                    validation = validate_proposer_content(content, selected_row)
                    attempts.append({
                        "attempt": attempt_index + 1,
                        "called_at_utc": datetime.now(timezone.utc).isoformat(),
                        "raw_response": raw_payload,
                        "content": content,
                        "validation_error": validation["validation_error"],
                    })
                    if not validation["validation_error"] or validation["validation_error"] == "EXPLICIT_AVOID_CONFLICT":
                        final_record = {
                            "query_id": query_id,
                            "bridge_phrase": selected_row["bridge_phrase"],
                            "attempts": attempts,
                            **validation,
                        }
                        break
                except Exception as error:
                    attempts.append({
                        "attempt": attempt_index + 1,
                        "called_at_utc": datetime.now(timezone.utc).isoformat(),
                        "api_error": repr(error),
                    })
                if attempt_index < MAX_RETRIES:
                    time.sleep(min(2 ** attempt_index, 4))
            if final_record is None:
                if attempts and "content" in attempts[-1]:
                    validation = validate_proposer_content(attempts[-1]["content"], selected_row)
                    final_record = {
                        "query_id": query_id,
                        "bridge_phrase": selected_row["bridge_phrase"],
                        "attempts": attempts,
                        **validation,
                    }
                else:
                    final_record = {
                        "query_id": query_id,
                        "bridge_phrase": selected_row["bridge_phrase"],
                        "attempts": attempts,
                        "completed": False,
                        "raw_bridge_status": "API_ERROR",
                        "validated_bridge_status": "",
                        "validated_targets": [],
                        "validation_error": "API_ERROR_AFTER_RETRIES",
                        "avoid_conflicts": [],
                    }
            checkpoint["llm_results"] = [
                item for item in checkpoint["llm_results"] if item["query_id"] != query_id
            ] + [final_record]
            write_json(CHECKPOINT_PATH, checkpoint)
        llm_ready = (
            len(checkpoint["llm_results"]) == 12
            and all(item.get("completed") for item in checkpoint["llm_results"])
        )
        if not llm_ready:
            print("LLM/API 호출이 모두 완료되지 않았습니다. checkpoint에서 재개할 수 있습니다.")
        else:
            print("12개 proposer 응답이 checkpoint에 고정되었습니다.")

if selection_validated and not RUN_LLM_ENABLED:
    print("READY_FOR_LLM: 12개 검증은 통과했지만 명시적 실행 gate가 닫혀 있어 API를 호출하지 않았습니다.")


12개 proposer 응답이 checkpoint에 고정되었습니다.


## 4. 고정 Retrieval과 blind pool 생성

아래 함수는 11번의 category score와 tie-break를 그대로 사용한다. Canonical Note 하나는
안전한 raw alias 중 하나라도 존재하면 1로 계산하고, 여러 canonical concept은 평균한다.
Baseline과 Treatment의 차이는 검증된 Bridge Accord/Canonical Note 추가뿐이다.


In [9]:
def build_retrieval_matrices():
    from scipy import sparse

    accord_names = accord_dictionary_df["accord"].astype(str).tolist()
    note_names = note_dictionary_df["note"].astype(str).tolist()
    accord_to_col = {name: index for index, name in enumerate(accord_names)}
    note_to_col = {name: index for index, name in enumerate(note_names)}

    perfume_ids = array.array("i")
    perfume_names, perfume_brands = [], []
    accord_rows, accord_cols, accord_values = array.array("i"), array.array("i"), array.array("f")
    note_rows, note_cols = array.array("i"), array.array("i")
    season_share_columns = {name: array.array("f") for name in ("winter", "spring", "summer", "autumn")}
    daypart_share_columns = {name: array.array("f") for name in ("day", "night")}

    with INPUTS["perfumes"].open("r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            record = json.loads(line)
            row_index = len(perfume_ids)
            perfume_ids.append(int(record["id"]))
            perfume_names.append(record.get("name") or "")
            perfume_brands.append(record.get("brand") or "")

            strength_by_col = {}
            for item in record.get("accords") or []:
                name, strength = item.get("name"), item.get("strength")
                if name not in accord_to_col or strength is None:
                    continue
                column, strength = accord_to_col[name], float(strength)
                if column not in strength_by_col or strength > strength_by_col[column]:
                    strength_by_col[column] = strength
            for column, strength in strength_by_col.items():
                accord_rows.append(row_index); accord_cols.append(column); accord_values.append(strength)

            notes = record.get("notes") or {}
            tiered = notes.get("tiered") or {}
            union_notes = set()
            for tier in ("top", "middle", "base"):
                union_notes.update(
                    item.get("name") for item in tiered.get(tier) or [] if item.get("name")
                )
            union_notes.update(item.get("name") for item in notes.get("flat") or [] if item.get("name"))
            for name in union_notes:
                if name in note_to_col:
                    note_rows.append(row_index); note_cols.append(note_to_col[name])

            seasons = record.get("seasons") or {}
            season_votes = np.asarray([float(seasons.get(name, 0) or 0) for name in ("winter", "spring", "summer", "autumn")])
            season_total = season_votes.sum()
            season_shares = season_votes / season_total if season_total >= 20 else np.full(4, np.nan)
            for name, value in zip(("winter", "spring", "summer", "autumn"), season_shares):
                season_share_columns[name].append(float(value))

            daypart = record.get("daypart") or {}
            daypart_votes = np.asarray([float(daypart.get(name, 0) or 0) for name in ("day", "night")])
            daypart_total = daypart_votes.sum()
            daypart_shares = daypart_votes / daypart_total if daypart_total >= 20 else np.full(2, np.nan)
            for name, value in zip(("day", "night"), daypart_shares):
                daypart_share_columns[name].append(float(value))

    n_perfumes = len(perfume_ids)
    accord_matrix = sparse.csr_matrix(
        (np.asarray(accord_values, dtype=np.float32),
         (np.asarray(accord_rows, dtype=np.int32), np.asarray(accord_cols, dtype=np.int32))),
        shape=(n_perfumes, len(accord_names)), dtype=np.float32,
    )
    note_matrix = sparse.csr_matrix(
        (np.ones(len(note_rows), dtype=np.float32),
         (np.asarray(note_rows, dtype=np.int32), np.asarray(note_cols, dtype=np.int32))),
        shape=(n_perfumes, len(note_names)), dtype=np.float32,
    )
    note_matrix.sum_duplicates(); note_matrix.data[:] = 1.0
    season_matrix = np.column_stack([
        np.asarray(season_share_columns[name], dtype=np.float32)
        for name in ("winter", "spring", "summer", "autumn")
    ])
    daypart_matrix = np.column_stack([
        np.asarray(daypart_share_columns[name], dtype=np.float32)
        for name in ("day", "night")
    ])
    return {
        "n_perfumes": n_perfumes,
        "perfume_ids": np.asarray(perfume_ids, dtype=np.int32),
        "perfume_names": perfume_names,
        "perfume_brands": perfume_brands,
        "accord_matrix": accord_matrix,
        "note_matrix": note_matrix,
        "season_matrix": season_matrix,
        "daypart_matrix": daypart_matrix,
        "accord_to_col": accord_to_col,
        "note_to_col": note_to_col,
        "season_to_col": {name: index for index, name in enumerate(("winter", "spring", "summer", "autumn"))},
        "daypart_to_col": {name: index for index, name in enumerate(("day", "night"))},
    }


def mean_sparse_columns(matrix, columns, scale=1.0):
    return np.asarray(matrix[:, columns].mean(axis=1)).ravel() / scale


def canonical_note_match(matrix, note_to_col, concepts):
    concept_arrays = []
    for concept in concepts:
        columns = [note_to_col[name] for name in safe_note_expansions.get(concept, []) if name in note_to_col]
        if not columns:
            raise ValueError(f"검색 가능한 safe raw alias가 없습니다: {concept}")
        matched = matrix[:, columns].max(axis=1)
        matched = np.asarray(matched.toarray()).ravel() if hasattr(matched, "toarray") else np.asarray(matched).ravel()
        concept_arrays.append((matched > 0).astype(np.float64))
    return np.vstack(concept_arrays).mean(axis=0)


def score_query(features, matrices):
    n_perfumes = matrices["n_perfumes"]
    category_arrays = {}
    if features["accords"]:
        columns = [matrices["accord_to_col"][name] for name in features["accords"]]
        category_arrays["accord_match"] = mean_sparse_columns(matrices["accord_matrix"], columns, 100.0)
    if features["canonical_notes"]:
        category_arrays["note_match"] = canonical_note_match(
            matrices["note_matrix"], matrices["note_to_col"], features["canonical_notes"]
        )
    if features["seasons"]:
        columns = [matrices["season_to_col"][name] for name in features["seasons"]]
        submatrix = matrices["season_matrix"][:, columns]
        reliable = np.isfinite(submatrix).all(axis=1)
        values = np.full(n_perfumes, np.nan)
        values[reliable] = submatrix[reliable].mean(axis=1)
        category_arrays["season_match"] = values
    if features["dayparts"]:
        columns = [matrices["daypart_to_col"][name] for name in features["dayparts"]]
        submatrix = matrices["daypart_matrix"][:, columns]
        reliable = np.isfinite(submatrix).all(axis=1)
        values = np.full(n_perfumes, np.nan)
        values[reliable] = submatrix[reliable].mean(axis=1)
        category_arrays["daypart_match"] = values
    if not category_arrays:
        return None
    stacked = np.vstack(list(category_arrays.values()))
    available_count = np.isfinite(stacked).sum(axis=0)
    base_match = np.divide(
        np.nansum(stacked, axis=0), available_count,
        out=np.full(n_perfumes, np.nan), where=available_count > 0,
    )
    evidence_coverage = available_count / len(category_arrays)
    final_score = np.where(available_count > 0, base_match * evidence_coverage, 0.0)
    return {"final_score": final_score, "base_match_score": base_match, "evidence_coverage": evidence_coverage}


def retrieve_top5(features, matrices):
    scores = score_query(features, matrices)
    if scores is None:
        return []
    top_k = min(TOP_K, matrices["n_perfumes"])
    candidate_rows = np.argpartition(-scores["final_score"], top_k - 1)[:top_k]
    order = np.lexsort((
        matrices["perfume_ids"][candidate_rows],
        -np.nan_to_num(scores["base_match_score"][candidate_rows], nan=-1.0),
        -scores["evidence_coverage"][candidate_rows],
        -scores["final_score"][candidate_rows],
    ))
    output = []
    for rank, row_index in enumerate(candidate_rows[order], start=1):
        output.append({
            "rank": rank,
            "perfume_id": int(matrices["perfume_ids"][row_index]),
            "name": matrices["perfume_names"][row_index],
            "brand": matrices["perfume_brands"][row_index],
            "final_score": float(scores["final_score"][row_index]),
            "base_match_score": float(scores["base_match_score"][row_index]),
            "evidence_coverage": float(scores["evidence_coverage"][row_index]),
        })
    return output


print("Retrieval functions loaded. perfumes.jsonl은 사전 선정 검증과 proposer 완료 후에만 읽습니다.")


Retrieval functions loaded. perfumes.jsonl은 사전 선정 검증과 proposer 완료 후에만 읽습니다.


In [10]:
def direct_features(selected_row):
    return {
        "accords": list(selected_row["direct_accords"]),
        "canonical_notes": list(selected_row["direct_canonical_notes"]),
        "seasons": list(selected_row["seasons"]),
        "dayparts": list(selected_row["dayparts"]),
    }


def treatment_features(selected_row, llm_result):
    features = direct_features(selected_row)
    for target in llm_result["validated_targets"]:
        key = "accords" if target["target_type"] == "ACCORD" else "canonical_notes"
        if target["target_feature"] not in features[key]:
            features[key].append(target["target_feature"])
    return features


def load_candidate_profiles(perfume_ids_needed):
    profiles = {}
    with INPUTS["perfumes"].open("r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            record = json.loads(line)
            perfume_id = int(record["id"])
            if perfume_id not in perfume_ids_needed:
                continue
            accords = sorted(
                [item for item in record.get("accords") or [] if item.get("name") and item.get("strength") is not None],
                key=lambda item: (-float(item["strength"]), item["name"]),
            )[:5]
            notes = record.get("notes") or {}
            tiered = notes.get("tiered") or {}
            profiles[perfume_id] = {
                "top_accords_json": json.dumps([
                    {"name": item["name"], "strength": float(item["strength"])} for item in accords
                ], ensure_ascii=False),
                "top_notes_json": json.dumps([item.get("name") for item in (tiered.get("top") or [])[:5] if item.get("name")], ensure_ascii=False),
                "middle_notes_json": json.dumps([item.get("name") for item in (tiered.get("middle") or [])[:5] if item.get("name")], ensure_ascii=False),
                "base_notes_json": json.dumps([item.get("name") for item in (tiered.get("base") or [])[:5] if item.get("name")], ensure_ascii=False),
                "flat_notes_json": json.dumps([item.get("name") for item in (notes.get("flat") or [])[:10] if item.get("name")], ensure_ascii=False),
                "seasons": record.get("seasons") or {},
                "daypart": record.get("daypart") or {},
            }
    if set(profiles) != set(perfume_ids_needed):
        raise RuntimeError("일부 pooled perfume profile을 찾지 못했습니다.")
    return profiles


glossary_lookup = {
    (row.feature_type, row.feature_name): {
        "definition": row.verified_definition,
        "aliases": json.loads(row.same_concept_aliases_json),
        "source": row.definition_source,
    }
    for row in glossary_df.itertuples(index=False)
}
canonical_by_raw = safe_map_df.drop_duplicates("raw_note").set_index("raw_note")["canonical_note"].to_dict()


def shown_glossary(profile):
    items = []
    for accord in json.loads(profile["top_accords_json"]):
        key = ("ACCORD", accord["name"])
        if key in glossary_lookup:
            items.append({"feature_type": key[0], "feature_name": key[1], **glossary_lookup[key]})
    raw_notes = []
    for column in ["top_notes_json", "middle_notes_json", "base_notes_json", "flat_notes_json"]:
        raw_notes.extend(json.loads(profile[column]))
    for raw_note in raw_notes:
        canonical = canonical_by_raw.get(raw_note)
        key = ("CANONICAL_NOTE", canonical)
        if canonical and key in glossary_lookup and not any(
            item["feature_type"] == key[0] and item["feature_name"] == key[1] for item in items
        ):
            items.append({"feature_type": key[0], "feature_name": key[1], **glossary_lookup[key]})
    return json.dumps(items, ensure_ascii=False)


def context_for_evaluation(profile, selected_row):
    output = {"seasons": {}, "dayparts": {}}
    season_votes = profile["seasons"]
    season_total = sum(float(season_votes.get(name, 0) or 0) for name in SEASONS)
    for name in selected_row["seasons"]:
        output["seasons"][name] = {
            "vote_eligible": season_total >= 20,
            "share": float(season_votes.get(name, 0) or 0) / season_total if season_total >= 20 else None,
        }
    daypart_votes = profile["daypart"]
    daypart_total = sum(float(daypart_votes.get(name, 0) or 0) for name in DAYPARTS)
    for name in selected_row["dayparts"]:
        output["dayparts"][name] = {
            "vote_eligible": daypart_total >= 20,
            "share": float(daypart_votes.get(name, 0) or 0) / daypart_total if daypart_total >= 20 else None,
        }
    return json.dumps(output, ensure_ascii=False)


def empty_human_fields():
    return {
        "r1_rating": "", "r2_rating": "", "adj_rating": "",
        "r1_abstain_cause": "", "r2_abstain_cause": "", "adj_abstain_cause": "",
        "r1_missed_allowed_targets_json": "", "r2_missed_allowed_targets_json": "", "adj_missed_allowed_targets_json": "",
        "r1_outside_target_candidates_json": "", "r2_outside_target_candidates_json": "", "adj_outside_target_candidates_json": "",
        "adjudication_note": "",
    }


human_table_ready = False
if llm_ready:
    selected_by_id = {row["query_id"]: row for row in preregistration["selection"]["selected_queries"]}
    llm_by_id = {row["query_id"]: row for row in checkpoint["llm_results"]}
    print("perfumes.jsonl을 한 번 스트리밍해 11번과 같은 Retrieval matrix를 만듭니다.")
    matrices = build_retrieval_matrices()
    retrieval_key = {}
    perfume_ids_needed = set()
    for query_id, selected_row in selected_by_id.items():
        baseline_features = direct_features(selected_row)
        treatment_feature_set = treatment_features(selected_row, llm_by_id[query_id])
        baseline = retrieve_top5(baseline_features, matrices)
        treatment = retrieve_top5(treatment_feature_set, matrices)
        perfume_ids_needed.update(item["perfume_id"] for item in baseline + treatment)
        retrieval_key[query_id] = {
            "baseline_features": baseline_features,
            "treatment_features": treatment_feature_set,
            "baseline": baseline,
            "treatment": treatment,
            "anonymous_candidates": [],
        }

    profiles = load_candidate_profiles(perfume_ids_needed)
    human_rows = []
    for query_id, selected_row in selected_by_id.items():
        llm_result = llm_by_id[query_id]
        if llm_result["validated_bridge_status"] == "MAPPED":
            for index, target in enumerate(llm_result["validated_targets"], start=1):
                glossary_item = glossary_lookup[(target["target_type"], target["target_feature"])]
                human_rows.append({
                    "task_type": "CONCEPT",
                    "query_id": query_id,
                    "resolution_type": selected_row["resolution_type"],
                    "primary_expression_type": selected_row["primary_expression_type"],
                    "bridge_evaluation_text": selected_row["bridge_evaluation_text"],
                    "anonymous_item_id": f"{query_id}-T{index:02d}",
                    "target_type": target["target_type"],
                    "target_feature": target["target_feature"],
                    "feature_definition": glossary_item["definition"],
                    "bridge_status": "MAPPED",
                    "top_accords_json": "", "top_notes_json": "", "middle_notes_json": "",
                    "base_notes_json": "", "flat_notes_json": "", "context_json": "",
                    "displayed_feature_glossary_json": "",
                    **empty_human_fields(),
                })
        else:
            human_rows.append({
                "task_type": "ABSTAIN_CAUSE",
                "query_id": query_id,
                "resolution_type": selected_row["resolution_type"],
                "primary_expression_type": selected_row["primary_expression_type"],
                "bridge_evaluation_text": selected_row["bridge_evaluation_text"],
                "anonymous_item_id": f"{query_id}-A01",
                "target_type": "", "target_feature": "", "feature_definition": "",
                "bridge_status": llm_result["validated_bridge_status"],
                "top_accords_json": "", "top_notes_json": "", "middle_notes_json": "",
                "base_notes_json": "", "flat_notes_json": "", "context_json": "",
                "displayed_feature_glossary_json": "",
                **empty_human_fields(),
            })

        pooled_ids = sorted({
            item["perfume_id"]
            for item in retrieval_key[query_id]["baseline"] + retrieval_key[query_id]["treatment"]
        })
        rng = random.Random(int(hashlib.sha256(f"{BLIND_SALT}{query_id}".encode()).hexdigest()[:16], 16))
        rng.shuffle(pooled_ids)
        baseline_rank = {item["perfume_id"]: item["rank"] for item in retrieval_key[query_id]["baseline"]}
        treatment_rank = {item["perfume_id"]: item["rank"] for item in retrieval_key[query_id]["treatment"]}
        details = {
            item["perfume_id"]: item
            for item in retrieval_key[query_id]["baseline"] + retrieval_key[query_id]["treatment"]
        }
        for index, perfume_id in enumerate(pooled_ids, start=1):
            anonymous_id = f"{query_id}-C{index:02d}"
            detail, profile = details[perfume_id], profiles[perfume_id]
            retrieval_key[query_id]["anonymous_candidates"].append({
                "anonymous_item_id": anonymous_id,
                "perfume_id": perfume_id,
                "name": detail["name"],
                "brand": detail["brand"],
                "baseline_rank": baseline_rank.get(perfume_id),
                "treatment_rank": treatment_rank.get(perfume_id),
                "baseline_detail": next((item for item in retrieval_key[query_id]["baseline"] if item["perfume_id"] == perfume_id), None),
                "treatment_detail": next((item for item in retrieval_key[query_id]["treatment"] if item["perfume_id"] == perfume_id), None),
            })
            human_rows.append({
                "task_type": "RETRIEVAL",
                "query_id": query_id,
                "resolution_type": selected_row["resolution_type"],
                "primary_expression_type": selected_row["primary_expression_type"],
                "bridge_evaluation_text": selected_row["bridge_evaluation_text"],
                "anonymous_item_id": anonymous_id,
                "target_type": "", "target_feature": "", "feature_definition": "",
                "bridge_status": "",
                "top_accords_json": profile["top_accords_json"],
                "top_notes_json": profile["top_notes_json"],
                "middle_notes_json": profile["middle_notes_json"],
                "base_notes_json": profile["base_notes_json"],
                "flat_notes_json": profile["flat_notes_json"],
                "context_json": context_for_evaluation(profile, selected_row),
                "displayed_feature_glossary_json": shown_glossary(profile),
                **empty_human_fields(),
            })

    expected_human_df = pd.DataFrame(human_rows)
    if HUMAN_EVAL_PATH.exists():
        human_eval_df = pd.read_csv(HUMAN_EVAL_PATH, dtype=str, keep_default_na=False)
        key_columns = [
            column for column in expected_human_df.columns
            if not column.startswith(("r1_", "r2_", "adj_")) and column != "adjudication_note"
        ]
        pd.testing.assert_frame_equal(
            human_eval_df[key_columns].reset_index(drop=True),
            expected_human_df[key_columns].astype(str).reset_index(drop=True),
        )
    else:
        expected_human_df.to_csv(HUMAN_EVAL_PATH, index=False, encoding="utf-8-sig")
        human_eval_df = pd.read_csv(HUMAN_EVAL_PATH, dtype=str, keep_default_na=False)
    checkpoint["retrieval_key"] = retrieval_key
    checkpoint["human_evaluation_path"] = str(HUMAN_EVAL_PATH.relative_to(ROOT))
    checkpoint["human_evaluation_blind_columns_verified"] = True
    write_json(CHECKPOINT_PATH, checkpoint)
    preregistration["state"] = "WAITING_FOR_HUMAN_EVALUATION"
    write_json(PREREG_PATH, preregistration)
    human_table_ready = True
    print(f"Blind human evaluation table: {HUMAN_EVAL_PATH} ({len(human_eval_df)} rows)")
    forbidden = {"perfume_id", "name", "brand", "method", "rank", "retrieval_score", "popularity", "reviews"}
    assert not forbidden.intersection(human_eval_df.columns)


perfumes.jsonl을 한 번 스트리밍해 11번과 같은 Retrieval matrix를 만듭니다.


Blind human evaluation table: C:\Users\SSAFY\Desktop\EDA\evaluation_data\semantic_bridge\22_pilot_human_evaluation.csv (70 rows)


## 5. 사람 평가 입력 안내

`22_pilot_human_evaluation.csv`가 생성되면 평가자 두 명에게 독립 복사본을 제공한다.
평가자는 서로의 열과 `analysis_outputs/22_semantic_bridge_pilot_checkpoint.json`을 보지 않는다.

- `CONCEPT`: `r1_rating`/`r2_rating`에 2(STRONG), 1(PLAUSIBLE), 0(WRONG/HARMFUL)
- `RETRIEVAL`: 동일 열에 2(HIGHLY_RELEVANT), 1(PARTIALLY_RELEVANT), 0(NOT_RELEVANT)
- `ABSTAIN_CAUSE`: 각 `*_abstain_cause`에 아래 하나
  - `APPROPRIATE_ABSTENTION`
  - `TARGET_SPACE_LIMITATION`
  - `MAPPING_FAILURE`
- `MAPPING_FAILURE`이면 `*_missed_allowed_targets_json`에 놓친 allowed target을
  `[{"target_type":"...","target_feature":"..."}]` 형식으로 하나 이상 기록한다.
- `TARGET_SPACE_LIMITATION`이면 `*_outside_target_candidates_json`에 진단용 후보 이름을 JSON list로 기록할 수 있다.
  기존 `fragrantica_note_canonical_map_v1.csv`의 전체 2,492 canonical concept 목록은 진단용으로만
  참고할 수 있으며, 기록한 outside 후보는 이번 Retrieval target에 추가되지 않는다.
- 독립 판정이 다를 때만 `adj_*`와 `adjudication_note`를 합의 review에서 채운다.

평가표에는 향수 ID·이름·브랜드·시스템·rank·score·popularity/review가 없다.
pooled candidate는 Query별 고정 seed로 섞였으며 실제 연결표는 checkpoint에만 있다.
`Pooled Human NDCG@5`는 이 작은 TOP 5 합집합 안의 상대 비교이며 전체 향수 corpus의 global NDCG가 아니다.


In [11]:
ABSTAIN_CAUSES = {"APPROPRIATE_ABSTENTION", "TARGET_SPACE_LIMITATION", "MAPPING_FAILURE"}


def resolved_human_value(row, field):
    left, right = clean(row[f"r1_{field}"]), clean(row[f"r2_{field}"])
    if left == right:
        return left
    adjudicated = clean(row[f"adj_{field}"])
    if not adjudicated:
        raise ValueError(f"{row['anonymous_item_id']} {field}: 불일치 합의값이 없습니다.")
    return adjudicated


def validate_missed_target_list(value):
    try:
        parsed = json.loads(clean(value))
        if not isinstance(parsed, list) or not parsed:
            return None
        normalized = []
        for target in parsed:
            if not isinstance(target, dict) or set(target) != {"target_type", "target_feature"}:
                return None
            target_type, target_feature = target["target_type"], target["target_feature"]
            if target_type not in ALLOWED_BY_TYPE or target_feature not in ALLOWED_BY_TYPE[target_type]:
                return None
            normalized.append((target_type, target_feature))
        if len(normalized) != len(set(normalized)):
            return None
        return sorted(normalized)
    except Exception:
        return None


def evaluation_issues(evaluation):
    issues = []
    for row in evaluation.to_dict("records"):
        task = row["task_type"]
        if task in {"CONCEPT", "RETRIEVAL"}:
            if clean(row["r1_rating"]) not in {"0", "1", "2"} or clean(row["r2_rating"]) not in {"0", "1", "2"}:
                issues.append({"anonymous_item_id": row["anonymous_item_id"], "issue": "두 rating 필요"})
            elif row["r1_rating"] != row["r2_rating"] and clean(row["adj_rating"]) not in {"0", "1", "2"}:
                issues.append({"anonymous_item_id": row["anonymous_item_id"], "issue": "rating 합의 필요"})
        elif task == "ABSTAIN_CAUSE":
            cause_1, cause_2 = clean(row["r1_abstain_cause"]), clean(row["r2_abstain_cause"])
            if cause_1 not in ABSTAIN_CAUSES or cause_2 not in ABSTAIN_CAUSES:
                issues.append({"anonymous_item_id": row["anonymous_item_id"], "issue": "두 abstain cause 필요"})
                continue
            adjudicated_cause = cause_1
            if cause_1 != cause_2:
                adjudicated_cause = clean(row["adj_abstain_cause"])
                if adjudicated_cause not in ABSTAIN_CAUSES:
                    issues.append({"anonymous_item_id": row["anonymous_item_id"], "issue": "abstain cause 합의 필요"})
                    continue
            missed_1 = validate_missed_target_list(row["r1_missed_allowed_targets_json"])
            missed_2 = validate_missed_target_list(row["r2_missed_allowed_targets_json"])
            if cause_1 == "MAPPING_FAILURE" and missed_1 is None:
                issues.append({"anonymous_item_id": row["anonymous_item_id"], "issue": "r1 missed allowed target 필요"})
            if cause_2 == "MAPPING_FAILURE" and missed_2 is None:
                issues.append({"anonymous_item_id": row["anonymous_item_id"], "issue": "r2 missed allowed target 필요"})
            if adjudicated_cause == "MAPPING_FAILURE":
                needs_adjudicated_targets = cause_1 != cause_2 or missed_1 != missed_2
                if needs_adjudicated_targets and validate_missed_target_list(row["adj_missed_allowed_targets_json"]) is None:
                    issues.append({"anonymous_item_id": row["anonymous_item_id"], "issue": "missed allowed target 합의 필요"})
    return issues


def weighted_kappa(left, right):
    left, right = np.asarray(left, dtype=int), np.asarray(right, dtype=int)
    if len(left) == 0:
        return np.nan
    if np.array_equal(left, right) and len(set(left)) == 1:
        return 1.0
    from sklearn.metrics import cohen_kappa_score
    return float(cohen_kappa_score(left, right, weights="quadratic"))


def dcg_at_5(ratings):
    return sum((2 ** value - 1) / math.log2(index + 2) for index, value in enumerate(ratings[:5]))


metrics_ready = False
if human_table_ready:
    issues = evaluation_issues(human_eval_df)
    if issues:
        print(f"WAITING_FOR_HUMAN_EVALUATION: 미완료/합의 필요 항목 {len(issues)}")
        display(pd.DataFrame(issues).head(30))
    else:
        evaluation = human_eval_df.copy()
        evaluation["resolved_rating"] = np.nan
        rating_mask = evaluation["task_type"].isin(["CONCEPT", "RETRIEVAL"])
        evaluation.loc[rating_mask, "resolved_rating"] = [
            int(resolved_human_value(row, "rating"))
            for row in evaluation.loc[rating_mask].to_dict("records")
        ]
        abstain_mask = evaluation["task_type"].eq("ABSTAIN_CAUSE")
        evaluation.loc[abstain_mask, "resolved_abstain_cause"] = [
            resolved_human_value(row, "abstain_cause")
            for row in evaluation.loc[abstain_mask].to_dict("records")
        ]

        concept = evaluation[evaluation["task_type"].eq("CONCEPT")].copy()
        retrieval = evaluation[evaluation["task_type"].eq("RETRIEVAL")].copy()
        abstains = evaluation[evaluation["task_type"].eq("ABSTAIN_CAUSE")].copy()
        supported_queries = set(concept.loc[concept["resolved_rating"].ge(1), "query_id"])
        appropriate_abstains = set(abstains.loc[
            abstains["resolved_abstain_cause"].eq("APPROPRIATE_ABSTENTION"), "query_id"
        ])
        safe_resolution_count = len(supported_queries | appropriate_abstains)
        graded_precision = float((concept["resolved_rating"] / 2).mean()) if len(concept) else np.nan
        harmful_rate = float(concept["resolved_rating"].eq(0).mean()) if len(concept) else np.nan

        retrieval_rating = retrieval.set_index(["query_id", "anonymous_item_id"])["resolved_rating"].to_dict()
        ndcg_by_query = {}
        mixed_deltas, mixed_wins = {}, 0
        pure_strong_hits = 0
        for query_id, key in checkpoint["retrieval_key"].items():
            selected_row = next(row for row in preregistration["selection"]["selected_queries"] if row["query_id"] == query_id)
            anonymous_by_perfume = {
                item["perfume_id"]: item["anonymous_item_id"] for item in key["anonymous_candidates"]
            }
            pooled_ratings = [
                int(retrieval_rating[(query_id, item["anonymous_item_id"])])
                for item in key["anonymous_candidates"]
            ]
            idcg = dcg_at_5(sorted(pooled_ratings, reverse=True))
            def ndcg(ranked):
                ratings = [int(retrieval_rating[(query_id, anonymous_by_perfume[item["perfume_id"]])]) for item in ranked]
                return dcg_at_5(ratings) / idcg if idcg > 0 else 0.0
            baseline_ndcg, treatment_ndcg = ndcg(key["baseline"]), ndcg(key["treatment"])
            ndcg_by_query[query_id] = {
                "resolution_type": selected_row["resolution_type"],
                "primary_expression_type": selected_row["primary_expression_type"],
                "baseline_pooled_human_ndcg_at_5": baseline_ndcg,
                "treatment_pooled_human_ndcg_at_5": treatment_ndcg,
                "delta": treatment_ndcg - baseline_ndcg,
            }
            if selected_row["resolution_type"] == "MIXED":
                mixed_deltas[query_id] = treatment_ndcg - baseline_ndcg
                mixed_wins += int(treatment_ndcg > baseline_ndcg)
            else:
                treatment_ratings = [
                    int(retrieval_rating[(query_id, anonymous_by_perfume[item["perfume_id"]])])
                    for item in key["treatment"]
                ]
                pure_strong_hits += int(any(value == 2 for value in treatment_ratings))

        concept_kappa = weighted_kappa(concept["r1_rating"], concept["r2_rating"])
        retrieval_kappa = weighted_kappa(retrieval["r1_rating"], retrieval["r2_rating"])
        concept_exact = float((concept["r1_rating"] == concept["r2_rating"]).mean()) if len(concept) else np.nan
        retrieval_exact = float((retrieval["r1_rating"] == retrieval["r2_rating"]).mean()) if len(retrieval) else np.nan
        mixed_mean_delta = float(np.mean(list(mixed_deltas.values()))) if mixed_deltas else np.nan

        def concept_group_diagnostics(frame, group_column):
            output = {}
            for group, values in frame.groupby(group_column, dropna=False):
                output[str(group)] = {
                    "target_count": int(len(values)),
                    "graded_score_mean": float((values["resolved_rating"] / 2).mean()),
                    "wrong_harmful_rate": float(values["resolved_rating"].eq(0).mean()),
                    "supported_query_count": int(values.loc[values["resolved_rating"].ge(1), "query_id"].nunique()),
                }
            return output

        def retrieval_group_diagnostics(group_column):
            grouped = {}
            values_df = pd.DataFrame.from_dict(ndcg_by_query, orient="index").reset_index(names="query_id")
            for group, values in values_df.groupby(group_column):
                grouped[str(group)] = {
                    "query_count": int(len(values)),
                    "mean_delta_pooled_human_ndcg_at_5": float(values["delta"].mean()),
                    "win_count": int(values["delta"].gt(0).sum()),
                    "tie_count": int(values["delta"].eq(0).sum()),
                    "loss_count": int(values["delta"].lt(0).sum()),
                }
            return grouped

        llm_status_rows = []
        for item in checkpoint["llm_results"]:
            selected_row = next(
                row for row in preregistration["selection"]["selected_queries"]
                if row["query_id"] == item["query_id"]
            )
            llm_status_rows.append({
                "query_id": item["query_id"],
                "resolution_type": selected_row["resolution_type"],
                "primary_expression_type": selected_row["primary_expression_type"],
                "validated_bridge_status": item["validated_bridge_status"],
            })
        llm_status_df = pd.DataFrame(llm_status_rows)
        status_counts = llm_status_df["validated_bridge_status"].value_counts().to_dict()
        status_by_expression_type = {
            str(group): values["validated_bridge_status"].value_counts().to_dict()
            for group, values in llm_status_df.groupby("primary_expression_type")
        }
        cause_counts = abstains["resolved_abstain_cause"].value_counts().to_dict()
        cause_rates = {
            key: value / len(abstains) for key, value in cause_counts.items()
        } if len(abstains) else {}
        mixed_ties = sum(value == 0 for value in mixed_deltas.values())
        mixed_losses = sum(value < 0 for value in mixed_deltas.values())
        context_only_pure = {
            query_id: values
            for query_id, values in ndcg_by_query.items()
            if values["resolution_type"] == "PURE"
            and checkpoint["retrieval_key"][query_id]["baseline"]
        }

        gate_pass = (
            np.isfinite(concept_kappa) and concept_kappa >= 0.40
            and np.isfinite(retrieval_kappa) and retrieval_kappa >= 0.40
        )
        go = (
            safe_resolution_count >= 9
            and np.isfinite(graded_precision) and graded_precision >= 0.67
            and np.isfinite(harmful_rate) and harmful_rate <= 0.20
            and mixed_wins >= 4 and mixed_mean_delta > 0
            and pure_strong_hits >= 4 and gate_pass
        )
        numeric_stop = (
            safe_resolution_count < 6
            or (np.isfinite(graded_precision) and graded_precision < 0.50)
            or (np.isfinite(harmful_rate) and harmful_rate > 0.30)
            or (np.isfinite(mixed_mean_delta) and mixed_mean_delta <= 0 and pure_strong_hits == 0)
        )
        decision = "INCONCLUSIVE" if not gate_pass else "GO" if go else "STOP" if numeric_stop else "REVISE"

        results = {
            "pilot_version": PILOT_VERSION,
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "decision": decision,
            "core_metrics": {
                "safe_resolution_count": safe_resolution_count,
                "safe_resolution_rate": safe_resolution_count / 12,
                "graded_concept_precision_at_3": graded_precision,
                "wrong_harmful_mapping_rate": harmful_rate,
                "mixed_mean_delta_pooled_human_ndcg_at_5": mixed_mean_delta,
                "mixed_win_count": mixed_wins,
                "pure_strong_hit_at_5_count": pure_strong_hits,
            },
            "diagnostics": {
                "supported_query_count": len(supported_queries),
                "supported_query_coverage": len(supported_queries) / 12,
                "validated_status_counts": status_counts,
                "validated_status_by_expression_type": status_by_expression_type,
                "abstain_cause_counts": cause_counts,
                "abstain_cause_rates_among_abstentions": cause_rates,
                "invalid_output_count": sum(
                    item["validation_error"] not in {"", "EXPLICIT_AVOID_CONFLICT"}
                    for item in checkpoint["llm_results"]
                ),
                "avoid_conflict_review_count": sum(
                    item["validation_error"] == "EXPLICIT_AVOID_CONFLICT"
                    for item in checkpoint["llm_results"]
                ),
                "concept_by_target_type": concept_group_diagnostics(concept, "target_type") if len(concept) else {},
                "concept_by_resolution_type": concept_group_diagnostics(concept, "resolution_type") if len(concept) else {},
                "concept_by_expression_type": concept_group_diagnostics(concept, "primary_expression_type") if len(concept) else {},
                "retrieval_by_resolution_type": retrieval_group_diagnostics("resolution_type"),
                "retrieval_by_expression_type": retrieval_group_diagnostics("primary_expression_type"),
                "mixed_delta_by_query": mixed_deltas,
                "mixed_tie_count": mixed_ties,
                "mixed_loss_count": mixed_losses,
                "context_only_pure_relative_results": context_only_pure,
                "pooled_human_ndcg_by_query": ndcg_by_query,
                "concept_exact_agreement": concept_exact,
                "retrieval_exact_agreement": retrieval_exact,
                "concept_quadratic_weighted_kappa": concept_kappa,
                "retrieval_quadratic_weighted_kappa": retrieval_kappa,
            },
            "pooled_ndcg_limitation": "Judged universe is each query's Baseline/Treatment TOP 5 union, not all perfumes.",
            "manual_stop_diagnostic_not_automated": "common-target stereotype convergence requires human error review",
        }
        write_json(RESULTS_PATH, results)
        preregistration["state"] = "COMPLETE"
        write_json(PREREG_PATH, preregistration)
        metrics_ready = True
        display(pd.DataFrame([results["core_metrics"] | {"decision": decision}]))
        print(f"Final results: {RESULTS_PATH}")


WAITING_FOR_HUMAN_EVALUATION: 미완료/합의 필요 항목 70


,anonymous_item_id,issue
0,SQ0061-A01,두 abstain cause 필요
1,SQ0032-A01,두 abstain cause 필요
2,SQ0032-C01,두 rating 필요
3,SQ0032-C02,두 rating 필요
4,SQ0032-C03,두 rating 필요
5,SQ0032-C04,두 rating 필요
6,SQ0032-C05,두 rating 필요
7,SQ0136-A01,두 abstain cause 필요
8,SQ0105-A01,두 abstain cause 필요
9,SQ0012-T01,두 rating 필요


In [12]:
if not selection_validated:
    CURRENT_STATE = "INVALID_PRESELECTED_QUERIES"
    NEXT_ACTION = "부적격 Query를 자동 교체하지 말고 selected CSV 작성자에게 검증 오류를 보고한다."
elif not RUN_LLM_ENABLED:
    CURRENT_STATE = "READY_FOR_LLM"
    NEXT_ACTION = (
        "12개 사전 선정과 annotation 검증 결과를 먼저 검토한다. 승인 후 RUN_LLM_ENABLED를 True로 바꾸고 "
        "같은 고정 12개로 Notebook을 재실행한다."
    )
elif not llm_ready:
    CURRENT_STATE = "WAITING_FOR_LLM_OR_API_RECOVERY"
    NEXT_ACTION = "GMS_KEY/API 상태를 확인하고 checkpoint resume로 같은 고정 12개만 완료한다."
elif not human_table_ready:
    CURRENT_STATE = "WAITING_FOR_BLIND_TABLE"
    NEXT_ACTION = "Notebook을 재실행해 blind human evaluation table 생성을 완료한다."
elif not metrics_ready:
    CURRENT_STATE = "WAITING_FOR_HUMAN_EVALUATION"
    NEXT_ACTION = (
        "두 평가자가 evaluation_data/semantic_bridge/22_pilot_human_evaluation.csv를 독립 평가하고, "
        "불일치만 합의한 뒤 Notebook을 재실행한다."
    )
else:
    CURRENT_STATE = "COMPLETE"
    NEXT_ACTION = "결과의 GO/REVISE/STOP 결정에 따라 후속 여부를 판단한다."

summary_lines = [
    f"## 현재 실행 상태: `{CURRENT_STATE}`",
    NEXT_ACTION,
    f"- Preselected 12-query validation: {'PASS' if selection_validated else 'FAIL'}",
    f"- LLM/API called in this run: {'YES' if RUN_LLM_ENABLED else 'NO'}",
    "- 전체 155개 성능으로의 일반화: 금지",
    "- Final Holdout 사용: 없음",
    "- Golden Set/survey 원본 수정: 없음",
    "- 신규 외부 데이터: 없음",
    "- FAMILY/RELATED/REVIEW/NOT_SAME의 SAME_CONCEPT 사용: 없음",
]
display(Markdown("\n\n".join(summary_lines)))


## 현재 실행 상태: `WAITING_FOR_HUMAN_EVALUATION`

두 평가자가 evaluation_data/semantic_bridge/22_pilot_human_evaluation.csv를 독립 평가하고, 불일치만 합의한 뒤 Notebook을 재실행한다.

- Preselected 12-query validation: PASS

- LLM/API called in this run: YES

- 전체 155개 성능으로의 일반화: 금지

- Final Holdout 사용: 없음

- Golden Set/survey 원본 수정: 없음

- 신규 외부 데이터: 없음

- FAMILY/RELATED/REVIEW/NOT_SAME의 SAME_CONCEPT 사용: 없음